# Classificação de Eficiência Energética — Estudo Acadêmico

Projeto desenvolvido para uma atividade prática da disciplina de **Inteligência Artificial e Computacional**, da **Universidade Feevale**. A versão original em notebook foi posteriormente evoluída com módulos reutilizáveis, CLI e testes automatizados.

Estudo do dataset **UCI Energy Efficiency (242)**. As classes são definidas pela mediana global de Y1 (carga de aquecimento), não por uma certificação energética oficial.

Instale o projeto conforme o README antes de executar. Execute as células em ordem. A regra original de mediana global e o holdout 70/30 foram preservados; consulte as limitações metodológicas no README.


In [ ]:
from pathlib import Path
from energy_efficiency.data import load_data
from energy_efficiency.experiment import run_experiment, comparison
from energy_efficiency.reporting import save_reports

# Use um caminho CSV local para executar sem acesso à UCI.
DATA_CSV = None
OUTPUT = Path("artifacts")


In [ ]:
X_original, y_original = load_data(DATA_CSV)
print(y_original.head())
print(f"Instâncias: {len(X_original)} | Atributos: {X_original.shape[1]}")
display(X_original.describe())


In [ ]:
experiment = run_experiment(X_original, y_original)
print(f"Mediana global de Y1: {experiment.threshold}")
print(f"Treino: {len(experiment.x_train)} | Teste: {len(experiment.x_test)}")
display(experiment.labels.value_counts().sort_index())
display(comparison(experiment))


In [ ]:
from sklearn.metrics import classification_report
for name, model in experiment.models.items():
    print(name)
    print(classification_report(experiment.y_test, model.predict(experiment.x_test),
          target_names=["Alta Eficiência (0)", "Baixa Eficiência (1)"], zero_division=0))
# O critério fica explícito e os empates são preservados.
table = comparison(experiment)
best = table.index[table["f1-score"] == table["f1-score"].max()].tolist()
print("Maior F1 macro nesta divisão:", ", ".join(best))


In [ ]:
from IPython.display import Image, display
output = save_reports(experiment, OUTPUT, source="local_csv" if DATA_CSV else "uci:242")
for image in sorted(output.glob("*.png")):
    display(Image(filename=str(image)))
print("Relatórios:", output.resolve())
